# 00 — Data and environment readiness
Run this notebook first. It validates the environment and dataset and displays sample images. It does not train models. The repository upload was incomplete at setup; a complete Drive copy can be used immediately.

### Environment and dataset configuration
**The one dataset path you may need to change is `DRIVE_DATA_ROOT` below.** It is an example, not a known shared-folder location. Point to the folder directly containing `Train.csv`, `Test.csv`, `Train/`, and `Test/`. If the folder is under "Shared with me", add a shortcut under My Drive first.

Colab defaults to Drive; local runs default to `Dataset/raw/`. Set `USE_DRIVE = False` to use repository data in Colab. Run cells in order. No API keys are needed.

In [ ]:
import sys
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

USE_DRIVE = IN_COLAB
# CHANGE THIS ONE DATASET PATH to your own shared Drive folder/shortcut:
DRIVE_DATA_ROOT = "/content/drive/MyDrive/CP467_Traffic_Signs/data/raw"
print("Running in Colab:", IN_COLAB)
print("Python:", sys.version)

### Mount Drive and locate project code
Opening a GitHub notebook in Colab does not clone its Python files. This cell finds a local project checkout or clones it once in Colab. For a code checkout stored on Drive, change the working directory to that checkout before this cell (see README).

In [ ]:
import os
import subprocess

if USE_DRIVE and IN_COLAB:
    drive.mount("/content/drive")

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "src" / "config.py").is_file() and (p / "requirements.txt").is_file()),
    None,
)
if PROJECT_ROOT is None and IN_COLAB:
    PROJECT_ROOT = Path("/content/CP467GroupProject")
    if not PROJECT_ROOT.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/Davy-Chu/CP467GroupProject.git", str(PROJECT_ROOT)],
            check=True,
        )
if PROJECT_ROOT is None or not (PROJECT_ROOT / "src" / "config.py").is_file():
    raise RuntimeError("Open from the project checkout or change directory to its root first.")

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)

### Dependencies
Colab often includes these libraries. Leave installation off first; if an import fails, set `INSTALL_DEPENDENCIES = True`, run this cell, restart the runtime if requested, then rerun from the top with installation off. Avoid replacing a working Colab GPU environment unnecessarily.

In [ ]:
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_ROOT / "requirements.txt")],
        check=True,
    )

import cv2
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import skimage
import sklearn
import torch
import torchvision
import seaborn

for library in (cv2, np, pd, matplotlib, skimage, sklearn, torch, torchvision, seaborn):
    print(f"{library.__name__}: {library.__version__}")

In [ ]:
from src.config import (
    NUM_CLASSES, IMAGE_SIZE, RANDOM_SEED, REPO_DATA_ROOT,
    RESULTS_DIR, MODELS_DIR, CONDITIONS, DEGRADATION_PARAMS,
    get_data_root, seed_everything,
)
from src.data_utils import (
    validate_dataset, load_metadata, load_image,
    make_train_validation_split, split_fingerprint,
)
from src.degradations import (
    apply_condition, reduce_brightness, add_noise, apply_motion_blur,
    equalize_histogram, gaussian_filter,
)

seed_everything()
DATA_ROOT = get_data_root(USE_DRIVE, DRIVE_DATA_ROOT)
print("Selected data root:", DATA_ROOT)
print("Repository fallback:", REPO_DATA_ROOT)
print("Shared conditions:", DEGRADATION_PARAMS)

## Dataset validation and shared split

Validation requires all 43 classes, the full 39,209/12,630 labeled image counts, and every referenced file. It fails if anything required is missing. The default decodes one image per class in each split; use `check_images="all"` for a slower full corruption scan on Drive. Do not disable the count checks to work around an unfinished upload.

In [ ]:
# This must pass before either team begins model development.
training_metadata, test_metadata = validate_dataset(DATA_ROOT, check_images="sample")
train_metadata, validation_metadata = make_train_validation_split(training_metadata)
SPLIT_ID = split_fingerprint(train_metadata, validation_metadata)
print(f"Training subset: {len(train_metadata):,}")
print(f"Validation subset: {len(validation_metadata):,}")
print(f"Official test (final evaluation only): {len(test_metadata):,}")
print("Split ID (must match across teammates):", SPLIT_ID)

## Dataset summary and class distribution

In [ ]:
summary = pd.DataFrame({
    "split": ["provided training", "training subset", "validation subset", "official test"],
    "images": [len(training_metadata), len(train_metadata), len(validation_metadata), len(test_metadata)],
    "classes": [frame["ClassId"].nunique() for frame in
                (training_metadata, train_metadata, validation_metadata, test_metadata)],
})
display(summary)
distribution = pd.DataFrame({
    "train": train_metadata["ClassId"].value_counts(),
    "validation": validation_metadata["ClassId"].value_counts(),
    "test": test_metadata["ClassId"].value_counts(),
}).sort_index()
display(distribution)
distribution[["train", "validation"]].plot.bar(stacked=True, figsize=(14, 4))
plt.xlabel("Class ID")
plt.ylabel("Images")
plt.title("Shared training/validation class distribution")
plt.tight_layout()
plt.show()

## Visual sanity check
A small sample from the provided training data; images are loaded as RGB and resized to the common size.

In [ ]:
samples = training_metadata.groupby("ClassId", sort=True).head(1).head(12)
fig, axes = plt.subplots(3, 4, figsize=(10, 7))
for ax, row in zip(axes.flat, samples.itertuples(index=False)):
    ax.imshow(load_image(DATA_ROOT, row.Path))
    ax.set_title(f"Class {row.ClassId}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## Shared degradation sanity check
Transforms are generated in memory. Both teams must call `apply_condition` with the CSV-relative image path before their model-specific preprocessing. Optional equalization/filtering are separate experiments.

In [ ]:
row = training_metadata.iloc[0]
sample_image = load_image(DATA_ROOT, row["Path"])
fig, axes = plt.subplots(1, len(CONDITIONS), figsize=(12, 3))
for ax, condition in zip(axes, CONDITIONS):
    ax.imshow(apply_condition(sample_image, condition, image_id=row["Path"]))
    ax.set_title(condition)
    ax.axis("off")
plt.tight_layout()
plt.show()
print("Readiness checks passed. Compare the split ID with your teammates.")
print("Open 01_hog_svm.ipynb or 02_cnn.ipynb to begin implementation.")